In [ ]:
import pandas as pd
import numpy as np
import re

from collections import Counter, defaultdict
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
DEV_PATH  = "../data/raw/development.csv"
SUB_PATH  = "submissionnoparese2.csv"
EVAL_PATH = "../data/raw/evaluation.csv"

MIN_RULE_SUPPORT = 30
MIN_RULE_PURITY  = 0.95
RULE_PRIORITY    = "best_purity_then_freq"

C_VALUE = 1.5
df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

for df in (df_dev, df_eval):
    df["article"] = df["article"].fillna("").astype(str)
    df["title"]   = df["title"].fillna("").astype(str)
    df["source"]  = df["source"].fillna("").astype(str)
def build_model_text(df):
    return (df["title"] + " " + df["article"]).str.lower()

df_dev["text"]  = build_model_text(df_dev)
df_eval["text"] = build_model_text(df_eval)
def add_numeric(df):
    df["n_tokens"]    = df["article"].str.split().str.len()
    df["title_len"]   = df["title"].str.len()
    df["article_len"] = df["article"].str.len()
    df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)
    return df

df_dev  = add_numeric(df_dev)
df_eval = add_numeric(df_eval)

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

for df in (df_dev, df_eval):
    df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)

FEATURES = ["source", "text"] + NUM_COLS

X_dev  = df_dev[FEATURES]
y_dev  = df_dev["label"].astype(int)
X_eval = df_eval[FEATURES]


def mine_pure_rules(texts, labels):
    counts = defaultdict(lambda: Counter())



    rule_token_to_class = {}
    rule_meta = {}

    for tok, c in counts.items():
        total = sum(c.values())
        if total < MIN_RULE_SUPPORT:
            continue

        best_class, best_freq = c.most_common(1)[0]
        purity = best_freq / total

        if purity >= MIN_RULE_PURITY:
            rule_token_to_class[tok] = int(best_class)
            rule_meta[tok] = (purity, total)

    return rule_token_to_class, rule_meta
def apply_rules(texts, rule_token_to_class, rule_meta):
    rule_pred = np.full(len(texts), -1, dtype=int)

    for i, txt in enumerate(texts):
        toks = set(tokenize_for_rules(txt))
        hits = [t for t in toks if t in rule_token_to_class]
        if not hits:
            continue

        hits.sort(
            key=lambda t: (rule_meta[t][0], rule_meta[t][1]),
            reverse=True
        )

        rule_pred[i] = rule_token_to_class[hits[0]]

    return rule_pred


def make_model():
    pre = ColumnTransformer(
        transformers=[
            ("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
            ("w_tfidf", TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.9,
                sublinear_tf=True,
                max_features=250_000
            ), "text"),
            ("c_tfidf", TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                min_df=3,
                max_df=0.9,
                sublinear_tf=True,
                max_features=300_000
            ), "text"),
            ("num", StandardScaler(), NUM_COLS),
        ],
        remainder="drop",
        n_jobs=-1
    )

    clf = LogisticRegression(
        C=C_VALUE,
        class_weight="balanced",
        max_iter=2000,
        n_jobs=-1
    )

    return Pipeline([
        ("pre", pre),
        ("clf", clf),
    ])
model = make_model()
model.fit(X_dev, y_dev)

rule_token_to_class, rule_meta = mine_pure_rules(
    df_dev["article"],
    y_dev
)

print("Rules mined:", len(rule_token_to_class))

model_pred = model.predict(X_eval)

rule_pred = apply_rules(
    df_eval["article"],
    rule_token_to_class,
    rule_meta
)

final_pred = model_pred.copy()
mask = rule_pred != -1
final_pred[mask] = rule_pred[mask]

print(f"Rule coverage on eval: {mask.mean():.3f}")


submission = pd.DataFrame({
    "Id": df_eval["Id"].astype(int),
    "Predicted": final_pred.astype(int)
})

submission.to_csv(SUB_PATH, index=False)
print("Saved:", SUB_PATH)       

Rules mined: 0
Rule coverage on eval: 0.000
Saved: submissionnoparese2.csv


In [12]:
SUB_PATH  = "submissionnoparese3.csv"
submission.to_csv(SUB_PATH, index=False)
print("Saved:", SUB_PATH)

Saved: submissionnoparese3.csv


In [13]:
submission.head()

,Id,Predicted
0,0,5
1,1,2
2,2,5
3,3,1
4,4,5


In [2]:
# ============================================================
# BASELINE — SINGLE-STAGE LINEAR MODEL (NO TWO-STAGE)
# - Regex HTML / URL friendly
# - Timestamp DROPPED
# - Train on FULL DEVELOPMENT
# - Predict on EVAL
# - Kaggle submission ready
# ============================================================

import pandas as pd
import numpy as np
import re

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# ============================================================
# PATHS (KAGGLE)
# ============================================================

DEV_PATH  = "../data/raw/development.csv"
SUB_PATH  = "baseline.csv"
EVAL_PATH = "../data/raw/evaluation.csv"

# ============================================================
# LOAD DATA
# ============================================================
df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

# ---- DROP TIMESTAMP COMPLETELY ----
df_dev  = df_dev.drop(columns=["timestamp"], errors="ignore")
df_eval = df_eval.drop(columns=["timestamp"], errors="ignore")

# ============================================================
# BASIC FIXES
# ============================================================
for df in (df_dev, df_eval):
    df["article"] = df["article"].fillna("").astype(str)
    df["title"]   = df["title"].fillna("").astype(str)
    df["source"]  = df["source"].fillna("").astype(str)

# ============================================================
# TEXT (JOINED)
# ============================================================
def build_text(df):
    return (df["title"] + " " + df["article"]).str.lower()

df_dev["text"]  = build_text(df_dev)
df_eval["text"] = build_text(df_eval)

# ============================================================
# NUMERIC FEATURES
# ============================================================
def add_numeric(df):
    df["n_tokens"]    = df["article"].str.split().str.len()
    df["title_len"]   = df["title"].str.len()
    df["article_len"] = df["article"].str.len()
    df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)
    return df

df_dev  = add_numeric(df_dev)
df_eval = add_numeric(df_eval)

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

for df in (df_dev, df_eval):
    df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)

# ============================================================
# FEATURES
# ============================================================
FEATURES = ["source", "text"] + NUM_COLS

X_dev  = df_dev[FEATURES]
y_dev  = df_dev["label"].astype(int)
X_eval = df_eval[FEATURES]

# ============================================================
# MODEL (NO RULES)
# ============================================================
pre = ColumnTransformer(
    transformers=[
        ("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),

        ("w_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.9,
            sublinear_tf=True,
            max_features=250_000
        ), "text"),

        ("c_tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=3,
            max_df=0.9,
            sublinear_tf=True,
            max_features=300_000
        ), "text"),

        ("num", StandardScaler(), NUM_COLS),
    ],
    remainder="drop",
    n_jobs=-1
)

model = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(
        C=1.5,
        class_weight="balanced",
        max_iter=2000,
        n_jobs=-1
    ))
])

# ============================================================
# TRAIN ON FULL DEVELOPMENT
# ============================================================
print("Training baseline model on full development...")
model.fit(X_dev, y_dev)

# ============================================================
# PREDICT ON EVAL
# ============================================================
print("Predicting on evaluation...")
final_pred = model.predict(X_eval)

# ============================================================
# SUBMISSION
# ============================================================
submission = pd.DataFrame({
    "Id": df_eval["Id"].astype(int),
    "Predicted": final_pred.astype(int)
})

submission.to_csv(SUB_PATH, index=False)
print("Submission saved to:", SUB_PATH)


Training baseline model on full development...
Predicting on evaluation...
Submission saved to: baseline.csv


In [3]:
row_by_id = df_dev[df_dev["Id"] == 310]
row_by_idx = df_dev.iloc[310]

print("=== BY Id ===")
print(row_by_id[["Id", "source", "title", "article"]])

print("\n=== BY iloc[310] ===")
print(row_by_idx[["Id", "source", "title", "article"]])

=== BY Id ===
      Id source                              title  \
310  310    BBC  Jaguar negotiates Â£534m Ford aid   

                                               article  
310  Ford will inject Â£534m into Jaguar, after the...  

=== BY iloc[310] ===
Id                                                       310
source                                                   BBC
title                      Jaguar negotiates Â£534m Ford aid
article    Ford will inject Â£534m into Jaguar, after the...
Name: 310, dtype: object


In [4]:
import re 
def has_broken_currency(text):
    return bool(re.search(r"Â[$£€¥]", text))

df_dev["broken_currency"] = df_dev["article"].apply(has_broken_currency)
df_dev["broken_currency"].mean()

np.float64(0.004325162193582259)

In [5]:
df_dev[df_dev["broken_currency"]].groupby("label").size().sort_values(ascending=False)


label
5    102
0     96
1     70
3     36
2     28
4     12
6      2
dtype: int64

In [6]:
def contains_currency_symbol(text):
    return any(sym in text for sym in ["£", "$", "€", "¥"])

df_dev["has_currency"] = df_dev["article"].apply(contains_currency_symbol)

df_dev.groupby("label")["has_currency"].mean().sort_values(ascending=False)


label
1    0.139686
2    0.058507
3    0.039591
5    0.035777
4    0.027059
0    0.024467
6    0.022566
Name: has_currency, dtype: float64

In [7]:
import re

CURRENCY_PATTERN = re.compile(
    r"(Â?[£$€¥])|\b(dollar|dollars|euro|euros|pound|pounds|yen)\b",
    flags=re.IGNORECASE
)

import re

CURRENCY_PATTERN = re.compile(
    r"(Â?[£$€¥])|\b(dollar|dollars|euro|euros|pound|pounds|yen)\b",
    flags=re.IGNORECASE
)
df_currency = df_dev[
    df_dev["article"].apply(lambda x: bool(CURRENCY_PATTERN.search(x)))
].copy()

print("Currency dataset size:", len(df_currency))
print("Percentage of dev:", len(df_currency) / len(df_dev))

df_currency["label"].value_counts(normalize=True).sort_index()
df_currency["label"].value_counts(normalize=True).sort_index()


Currency dataset size: 4995
Percentage of dev: 0.06243984149405603


label
0    0.192392
1    0.371371
2    0.152352
3    0.097497
4    0.054454
5    0.110310
6    0.021622
Name: proportion, dtype: float64

In [8]:
df_currency["label"].value_counts(normalize=True).sort_index()


label
0    0.192392
1    0.371371
2    0.152352
3    0.097497
4    0.054454
5    0.110310
6    0.021622
Name: proportion, dtype: float64

In [10]:
df_dev["label"].value_counts(normalize=True).sort_index()



label
0    0.294286
1    0.132355
2    0.139518
3    0.124717
4    0.107179
5    0.163169
6    0.038776
Name: proportion, dtype: float64

In [11]:
from collections import Counter

label_counts = Counter(df_currency["label"])
label_counts
total = sum(label_counts.values())
{lbl: cnt / total for lbl, cnt in label_counts.items()}


{0: 0.1923923923923924,
 5: 0.11031031031031031,
 3: 0.0974974974974975,
 2: 0.15235235235235237,
 1: 0.3713713713713714,
 4: 0.05445445445445445,
 6: 0.021621621621621623}

In [16]:
# ============================================================
# BASELINE LINEAR MODEL — CURRENCY FIX + REGEX (NO RULES)
# ============================================================

import pandas as pd
import numpy as np
import re

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# =========================
# CONFIG
# =========================

DEV_PATH  = "../data/raw/development.csv"
SUB_PATH  = "submission_baseline_currency.csv"
EVAL_PATH = "../data/raw/evaluation.csv"

C_VALUE = 1.5

# =========================
# LOAD
# =========================
df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

# =========================
# BASIC FIXES
# =========================
for df in (df_dev, df_eval):
    df["article"] = df["article"].fillna("").astype(str)
    df["title"]   = df["title"].fillna("").astype(str)
    df["source"]  = df["source"].fillna("").astype(str)

# =========================
# TEXT CLEANING (MINIMAL & SAFE)
# =========================
def fix_encoding(text):
    try:
        return text.encode("latin1").decode("utf-8")
    except:
        return text

def normalize_currency(text):
    return re.sub(
        r"[£$€¥]\s*\d+(\.\d+)?\s*(m|bn|million|billion)?",
        "CURRENCY_AMOUNT",
        text,
        flags=re.I
    )

def clean_text(text):
    text = fix_encoding(text)
    text = normalize_currency(text)
    return text.lower()

# =========================
# BUILD TEXT
# =========================
for df in (df_dev, df_eval):
    df["text"] = (df["title"] + " " + df["article"]).apply(clean_text)

# =========================
# NUMERIC FEATURES
# =========================
def add_numeric(df):
    df["n_tokens"]    = df["article"].str.split().str.len()
    df["title_len"]   = df["title"].str.len()
    df["article_len"] = df["article"].str.len()
    df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)
    return df

df_dev  = add_numeric(df_dev)
df_eval = add_numeric(df_eval)

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

for df in (df_dev, df_eval):
    df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)

# =========================
# FEATURES
# =========================
FEATURES = ["source", "text"] + NUM_COLS

X_dev  = df_dev[FEATURES]
y_dev  = df_dev["label"].astype(int)
X_eval = df_eval[FEATURES]

# =========================
# MODEL
# =========================
pre = ColumnTransformer(
    transformers=[
        ("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
        ("w", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1,2),
            min_df=3,
            max_df=0.9,
            sublinear_tf=True,
            max_features=250_000,
            token_pattern=r"[a-z0-9_:/\.]+"
        ), "text"),
        ("c", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3,5),
            min_df=3,
            max_df=0.9,
            sublinear_tf=True,
            max_features=300_000
        ), "text"),
        ("n", StandardScaler(), NUM_COLS),
    ],
    remainder="drop",
    n_jobs=-1
)

model = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(
        C=C_VALUE,
        class_weight="balanced",
        max_iter=2000,
        n_jobs=-1
    ))
])

# =========================
# TRAIN
# =========================
print("Training baseline model...")
model.fit(X_dev, y_dev)

# =========================
# PREDICT
# =========================
pred = model.predict(X_eval)

# =========================
# SUBMISSION
# =========================
submission = pd.DataFrame({
    "Id": df_eval["Id"].astype(int),
    "Predicted": pred.astype(int)
})

submission.to_csv(SUB_PATH, index=False)
print("Saved:", SUB_PATH)



Training baseline model...
Saved: submission_baseline_currency.csv


In [21]:
df = pd.read_csv(DEV_PATH)
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)

# testo completo (come nel modello baseline)
df["text"] = (df["title"] + " " + df["article"]).str.lower()

In [22]:
# regex: tutto ciò che NON è lettera, spazio o backslash
non_alpha_regex = re.compile(r"[^a-zA-Z\\ ]")

def count_non_alpha(text):
	return len(non_alpha_regex.findall(text))

df["non_alpha_chars"] = df["text"].apply(count_non_alpha)
df["text_len"] = df["text"].str.len()

# normalizziamo anche per lunghezza
df["non_alpha_ratio"] = df["non_alpha_chars"] / (df["text_len"] + 1)

agg = df.groupby("label").agg(
	mean_non_alpha=("non_alpha_chars", "mean"),
	median_non_alpha=("non_alpha_chars", "median"),
	mean_ratio=("non_alpha_ratio", "mean"),
	median_ratio=("non_alpha_ratio", "median"),
	count=("label", "count")
)

agg

df.groupby("label")["non_alpha_ratio"].describe()


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,23542.0,0.064976,0.065837,0.0,0.023585,0.037190,0.067164,0.386861
1,10588.0,0.066832,0.073914,0.0,0.024820,0.041971,0.071675,0.433198
2,11161.0,0.080503,0.078155,0.0,0.028409,0.048148,0.096774,0.473373
3,9977.0,0.065736,0.060439,0.0,0.028169,0.046218,0.073930,0.326007
4,8574.0,0.067739,0.060060,0.0,0.030120,0.047393,0.076495,0.349206
5,13053.0,0.061565,0.071830,0.0,0.021583,0.035714,0.061372,0.379310
6,3102.0,0.045616,0.042375,0.0,0.022035,0.033988,0.052448,0.338816


In [23]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

X = df[["non_alpha_ratio"]]
y = df["label"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for tr, te in skf.split(X, y):
	clf = LogisticRegression(max_iter=1000, class_weight="balanced")
	clf.fit(X.iloc[tr], y.iloc[tr])
	pred = clf.predict(X.iloc[te])
	scores.append(f1_score(y.iloc[te], pred, average="macro"))

np.mean(scores)


np.float64(0.07399728425851884)

In [ ]:
import re

URL_REGEX = re.compile(
    r"(http?://[^\s\"\'<>]+)",
    flags=re.IGNORECASE
)

def extract_urls(text):
    return URL_REGEX.findall(text)

df_dev["urls"] = df_dev["article"].apply(extract_urls)
df_dev["n_urls"] = df_dev["urls"].str.len()
df_dev.groupby("label")["n_urls"].describe()



,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,23542.0,0.448602,1.220120,0.0,0.0,0.0,0.0,28.0
1,10588.0,0.342180,1.023816,0.0,0.0,0.0,0.0,12.0
2,11161.0,1.216647,3.133501,0.0,0.0,0.0,0.0,56.0
3,9977.0,0.395109,2.026352,0.0,0.0,0.0,0.0,65.0
4,8574.0,0.252508,0.747508,0.0,0.0,0.0,0.0,9.0
5,13053.0,0.438980,1.859834,0.0,0.0,0.0,0.0,13.0
6,3102.0,0.153449,0.804886,0.0,0.0,0.0,0.0,9.0


In [35]:
import re

URL_REGEX = re.compile(
    r"(https?://[^\s\"\'<>]+)",
    flags=re.IGNORECASE
)

def extract_urls(text):
    return URL_REGEX.findall(text)

df_dev["urls"] = df_dev["article"].apply(extract_urls)
df_dev["n_urls"] = df_dev["urls"].str.len()
df_dev.groupby("label")["n_urls"].describe()



,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,23542.0,0.448602,1.220120,0.0,0.0,0.0,0.0,28.0
1,10588.0,0.342180,1.023816,0.0,0.0,0.0,0.0,12.0
2,11161.0,1.218170,3.134639,0.0,0.0,0.0,0.0,56.0
3,9977.0,0.395109,2.026352,0.0,0.0,0.0,0.0,65.0
4,8574.0,0.252508,0.747508,0.0,0.0,0.0,0.0,9.0
5,13053.0,0.438980,1.859834,0.0,0.0,0.0,0.0,13.0
6,3102.0,0.153449,0.804886,0.0,0.0,0.0,0.0,9.0


In [36]:
import re

URL_REGEX = re.compile(
    r"(https?://[^\s\"\'<>]+)",
    flags=re.IGNORECASE
)

def extract_urls(text):
    return URL_REGEX.findall(text)

df_dev["urls"] = df_dev["article"].apply(extract_urls)
df_dev["n_urls"] = df_dev["urls"].str.len()
df_dev.groupby("label")["n_urls"].describe()



,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,23542.0,0.448602,1.220120,0.0,0.0,0.0,0.0,28.0
1,10588.0,0.342180,1.023816,0.0,0.0,0.0,0.0,12.0
2,11161.0,1.218170,3.134639,0.0,0.0,0.0,0.0,56.0
3,9977.0,0.395109,2.026352,0.0,0.0,0.0,0.0,65.0
4,8574.0,0.252508,0.747508,0.0,0.0,0.0,0.0,9.0
5,13053.0,0.438980,1.859834,0.0,0.0,0.0,0.0,13.0
6,3102.0,0.153449,0.804886,0.0,0.0,0.0,0.0,9.0


In [28]:
def extract_domain(url):
    m = re.search(r"http?://([^/]+)/?", url)
    return m.group(1).lower() if m else None

df_dev["domains"] = df_dev["urls"].apply(
    lambda urls: [extract_domain(u) for u in urls]
)
dom_df = df_dev[["label", "domains"]].explode("domains")
dom_df = dom_df.dropna()



In [29]:
from collections import Counter

domain_stats = {}

for dom, g in dom_df.groupby("domains"):
    cnt = Counter(g["label"])
    total = sum(cnt.values())
    best_label, best_freq = cnt.most_common(1)[0]
    purity = best_freq / total
    if total >= 30 and purity >= 0.9:
        domain_stats[dom] = (best_label, purity, total)

len(domain_stats)


13

In [37]:
def extract_domain(url):
    m = re.search(r"https?://([^/]+)/?", url)
    return m.group(1).lower() if m else None

df_dev["domains"] = df_dev["urls"].apply(
    lambda urls: [extract_domain(u) for u in urls]
)
dom_df = df_dev[["label", "domains"]].explode("domains")
dom_df = dom_df.dropna()

from collections import Counter

domain_stats = {}

for dom, g in dom_df.groupby("domains"):
    cnt = Counter(g["label"])
    total = sum(cnt.values())
    best_label, best_freq = cnt.most_common(1)[0]
    purity = best_freq / total
    if total >= 30 and purity >= 0.9:
        domain_stats[dom] = (best_label, purity, total)

len(domain_stats)



13

In [38]:
def url_digit_ratio(url):
    digits = sum(c.isdigit() for c in url)
    return digits / max(len(url), 1)

df_dev["url_digit_ratio"] = df_dev["urls"].apply(
    lambda urls: max([url_digit_ratio(u) for u in urls], default=0)
)

In [30]:
def url_digit_ratio(url):
    digits = sum(c.isdigit() for c in url)
    return digits / max(len(url), 1)

df_dev["url_digit_ratio"] = df_dev["urls"].apply(
    lambda urls: max([url_digit_ratio(u) for u in urls], default=0)
)


In [31]:
df_dev.groupby("label")["url_digit_ratio"].describe()


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,23542.0,0.047219,0.099572,0.0,0.0,0.0,0.0,0.555556
1,10588.0,0.025982,0.077193,0.0,0.0,0.0,0.0,0.376623
2,11161.0,0.038392,0.080633,0.0,0.0,0.0,0.0,0.415385
3,9977.0,0.030954,0.084311,0.0,0.0,0.0,0.0,0.384615
4,8574.0,0.028776,0.078755,0.0,0.0,0.0,0.0,0.325397
5,13053.0,0.025503,0.076541,0.0,0.0,0.0,0.0,0.376623
6,3102.0,0.013283,0.055720,0.0,0.0,0.0,0.0,0.317568


In [32]:
def clean_urls_light(text):
    return re.sub(
        r"https?://[^\s\"\'<>]{60,}",
        " URL_LONG ",
        text
    )

df_dev["article_clean"] = df_dev["article"].apply(clean_urls_light)
df_eval["article_clean"] = df_eval["article"].apply(clean_urls_light)


In [33]:
TAG_REGEX = re.compile(r"<\s*([a-zA-Z0-9]+)")

def extract_tags(text):
    return TAG_REGEX.findall(text.lower())

df_dev["tags"] = df_dev["article"].apply(extract_tags)


In [34]:
tag_df = df_dev[["label", "tags"]].explode("tags")
tag_df = tag_df.dropna()

tag_stats = {}

for tag, g in tag_df.groupby("tags"):
    cnt = Counter(g["label"])
    total = sum(cnt.values())
    best_label, best_freq = cnt.most_common(1)[0]
    purity = best_freq / total
    if total >= 50 and purity >= 0.9:
        tag_stats[tag] = (best_label, purity, total)

tag_stats


{'em': (2, 0.9384615384615385, 65),
 'font': (2, 0.9285714285714286, 84),
 'h4': (2, 1.0, 628),
 'strong': (2, 1.0, 135)}

In [39]:
# ============================
# ANALISI URL + META TAG HTML
# Dataset: ../data/raw/development.csv
# ============================

import pandas as pd
import re
from collections import Counter, defaultdict

# ----------------------------
# LOAD DATA
# ----------------------------
PATH = "../data/raw/development.csv"

df = pd.read_csv(PATH)
df["article"] = df["article"].fillna("")
df["label"] = df["label"].astype(int)

print("Dataset shape:", df.shape)
print("Labels:", sorted(df["label"].unique()))

# ----------------------------
# REGEX DEFINITIONS
# ----------------------------
URL_REGEX = re.compile(
	r'https?://[^\s"<>\]]+',
	flags=re.IGNORECASE
)

HTML_TAG_REGEX = re.compile(
	r'<\s*([a-zA-Z0-9]+)(\s+[^>]*)?>',
	flags=re.IGNORECASE
)

# ----------------------------
# URL FEATURE EXTRACTION
# ----------------------------
def extract_url_features(text):
	urls = URL_REGEX.findall(text)

	n_urls = len(urls)
	n_http = sum(u.lower().startswith("http://") for u in urls)
	n_https = sum(u.lower().startswith("https://") for u in urls)

	n_digits = sum(len(re.findall(r'\d', u)) for u in urls)
	avg_len = sum(len(u) for u in urls) / n_urls if n_urls > 0 else 0

	return pd.Series({
		"url_count": n_urls,
		"http_count": n_http,
		"https_count": n_https,
		"url_digits": n_digits,
		"url_avg_len": avg_len
	})

url_feats = df["article"].apply(extract_url_features)
df_url = pd.concat([df[["label"]], url_feats], axis=1)

print("\n--- URL FEATURES (sample) ---")
print(df_url.head())

# ----------------------------
# URL STATS PER LABEL
# ----------------------------
url_stats_by_label = (
	df_url
	.groupby("label")
	.agg(["mean", "std", "median"])
)

print("\n--- URL STATS BY LABEL ---")
print(url_stats_by_label)

# ----------------------------
# HTML TAG EXTRACTION
# ----------------------------
def extract_html_tags(text):
	return [m.group(1).lower() for m in HTML_TAG_REGEX.finditer(text)]

tag_counts_by_label = defaultdict(Counter)

for label, text in zip(df["label"], df["article"]):
	tags = extract_html_tags(text)
	tag_counts_by_label[label].update(tags)

tag_df = (
	pd.DataFrame(tag_counts_by_label)
	.fillna(0)
	.T
)

label_counts = df["label"].value_counts().sort_index()
tag_df_norm = tag_df.div(label_counts, axis=0)

print("\n--- HTML TAG DISTRIBUTION (normalized, sample) ---")
print(tag_df_norm.iloc[:, :10].head())

# ----------------------------
# TOP TAGS PER LABEL
# ----------------------------
TOP_K = 15

for label in sorted(tag_df_norm.index):
	print(f"\n=== TOP HTML TAGS | LABEL {label} ===")
	print(
		tag_df_norm
		.loc[label]
		.sort_values(ascending=False)
		.head(TOP_K)
	)

# ----------------------------
# OPTIONAL: FEATURE MATRICES
# ----------------------------
# Normalized numeric features (ready to join)
html_features = tag_df_norm.add_prefix("html_tag_")

# Binary presence features
html_features_binary = (tag_df > 0).astype(int).add_prefix("html_tag_bin_")

print("\nHTML feature matrices ready:")
print(" - html_features (normalized)")
print(" - html_features_binary (binary)")

# ----------------------------
# SUMMARY
# ----------------------------
print("\nDONE.")
print("Extracted:")
print("- URL structural features (http/https, digits, length)")
print("- HTML meta-tag distributions (no hardcoding)")
print("- Label-conditional statistics")


Dataset shape: (79997, 7)
Labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

--- URL FEATURES (sample) ---
   label  url_count  http_count  https_count  url_digits  url_avg_len
0      5        0.0         0.0          0.0         0.0          0.0
1      0        0.0         0.0          0.0         0.0          0.0
2      0        0.0         0.0          0.0         0.0          0.0
3      0        0.0         0.0          0.0         0.0          0.0
4      0        2.0         2.0          0.0        34.0        108.5

--- URL STATS BY LABEL ---
      url_count                  http_count                  https_count  \
           mean       std median       mean       std median        mean   
label                                                                      
0      0.448602  1.220120    0.0   0.448602  1.220120    0.0    0.000000   
1      0.342180  1.023816    0.0   0.342180  1.023816    0.0    0.000000   
2      1.218170

In [40]:
# ============================
# ANALISI PAROLE CON MAIUSCOLO
# Dataset: ../data/raw/development.csv
# ============================

import pandas as pd
import re
from collections import defaultdict, Counter

# ----------------------------
# LOAD DATA
# ----------------------------
PATH = "../data/raw/development.csv"

df = pd.read_csv(PATH)
df["article"] = df["article"].fillna("")
df["label"] = df["label"].astype(int)

print("Dataset shape:", df.shape)
print("Labels:", sorted(df["label"].unique()))

# ----------------------------
# REGEX DEFINITIONS
# ----------------------------

# Parole tipo: iPhone, Google, HTTPServer, AIModel
CAPITALIZED_WORD_REGEX = re.compile(
	r'\b[A-Z][a-zA-Z0-9]{2,}\b'
)

# Parole tipo: HTTP, API, JSON, GPU
ALL_CAPS_WORD_REGEX = re.compile(
	r'\b[A-Z]{2,}\b'
)

# ----------------------------
# FEATURE EXTRACTION
# ----------------------------
def extract_caps_features(text):
	capitalized = CAPITALIZED_WORD_REGEX.findall(text)
	all_caps = ALL_CAPS_WORD_REGEX.findall(text)

	return pd.Series({
		"capitalized_count": len(capitalized),
		"all_caps_count": len(all_caps),
		"unique_capitalized": len(set(capitalized)),
		"unique_all_caps": len(set(all_caps)),
	})

caps_feats = df["article"].apply(extract_caps_features)
df_caps = pd.concat([df[["label"]], caps_feats], axis=1)

print("\n--- CAPS FEATURES (sample) ---")
print(df_caps.head())

# ----------------------------
# STATS PER LABEL
# ----------------------------
caps_stats_by_label = (
	df_caps
	.groupby("label")
	.agg(["mean", "std", "median"])
)

print("\n--- CAPS STATS BY LABEL ---")
print(caps_stats_by_label)

# ----------------------------
# VOCABOLARIO CAPS PER LABEL
# ----------------------------
caps_vocab_by_label = defaultdict(Counter)

for label, text in zip(df["label"], df["article"]):
	caps_vocab_by_label[label].update(
		CAPITALIZED_WORD_REGEX.findall(text)
	)
	caps_vocab_by_label[label].update(
		ALL_CAPS_WORD_REGEX.findall(text)
	)

# ----------------------------
# TOP CAPS WORDS PER LABEL
# ----------------------------
TOP_K = 20

for label in sorted(caps_vocab_by_label.keys()):
	print(f"\n=== TOP CAPS WORDS | LABEL {label} ===")
	for word, cnt in caps_vocab_by_label[label].most_common(TOP_K):
		print(f"{word:20s} {cnt}")

# ----------------------------
# PRESENZA CAPS COME TRIGGER
# ----------------------------
df_caps["has_caps"] = (
	(df_caps["capitalized_count"] > 0) |
	(df_caps["all_caps_count"] > 0)
).astype(int)

caps_presence_by_label = (
	df_caps
	.groupby("label")["has_caps"]
	.mean()
)

print("\n--- P(has_caps | label) ---")
print(caps_presence_by_label)

# ----------------------------
# SUMMARY
# ----------------------------
print("\nDONE.")
print("Extracted:")
print("- capitalized words (CamelCase / ProperCase)")
print("- ALL CAPS tokens (API, HTTP, GPU, …)")
print("- label-conditional statistics")
print("- caps presence as deterministic trigger")


Dataset shape: (79997, 7)
Labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

--- CAPS FEATURES (sample) ---
   label  capitalized_count  all_caps_count  unique_capitalized  \
0      5                  8               2                   8   
1      0                  6               0                   6   
2      0                  4               0                   4   
3      0                  1               0                   1   
4      0                 12               1                  11   

   unique_all_caps  
0                2  
1                0  
2                0  
3                0  
4                1  

--- CAPS STATS BY LABEL ---
      capitalized_count                   all_caps_count                   \
                   mean        std median           mean       std median   
label                                                                       
0              8.343981   7.406465    6.0       1.026

In [41]:
COUNTRIES = [
	"Afghanistan","Albania","Algeria","Andorra","Angola","Antigua","Argentina",
	"Armenia","Australia","Austria","Azerbaijan","Bahamas","Bahrain","Bangladesh",
	"Barbados","Belarus","Belgium","Belize","Benin","Bhutan","Bolivia",
	"Bosnia","Botswana","Brazil","Brunei","Bulgaria","Burkina","Burundi",
	"Cambodia","Cameroon","Canada","Chad","Chile","China","Colombia","Comoros",
	"Congo","Costa Rica","Croatia","Cuba","Cyprus","Czech","Denmark","Djibouti",
	"Dominica","Ecuador","Egypt","El Salvador","Estonia","Ethiopia","Fiji",
	"Finland","France","Gabon","Gambia","Georgia","Germany","Ghana","Greece",
	"Grenada","Guatemala","Guinea","Guyana","Haiti","Honduras","Hungary",
	"Iceland","India","Indonesia","Iran","Iraq","Ireland","Israel","Italy",
	"Jamaica","Japan","Jordan","Kazakhstan","Kenya","Kuwait","Latvia","Lebanon",
	"Lesotho","Liberia","Libya","Liechtenstein","Lithuania","Luxembourg",
	"Madagascar","Malawi","Malaysia","Maldives","Mali","Malta","Mauritania",
	"Mauritius","Mexico","Moldova","Monaco","Mongolia","Montenegro","Morocco",
	"Mozambique","Myanmar","Namibia","Nepal","Netherlands","New Zealand",
	"Nicaragua","Niger","Nigeria","Norway","Oman","Pakistan","Panama","Paraguay",
	"Peru","Philippines","Poland","Portugal","Qatar","Romania","Russia","Rwanda",
	"Saudi Arabia","Senegal","Serbia","Singapore","Slovakia","Slovenia",
	"Somalia","South Africa","Spain","Sri Lanka","Sudan","Suriname","Sweden",
	"Switzerland","Syria","Taiwan","Tajikistan","Tanzania","Thailand","Togo",
	"Tonga","Tunisia","Turkey","Turkmenistan","Uganda","Ukraine",
	"United Arab Emirates","United Kingdom","United States","Uruguay",
	"Uzbekistan","Venezuela","Vietnam","Yemen","Zambia","Zimbabwe"
]
import re

COUNTRY_REGEX = re.compile(
	r'\b(?:' + '|'.join(re.escape(c) for c in COUNTRIES) + r')\b',
	flags=re.IGNORECASE
)
import pandas as pd

# ----------------------------
# LOAD DATA
# ----------------------------
PATH = "../data/raw/development.csv"
df = pd.read_csv(PATH)
df["article"] = df["article"].fillna("")
df["label"] = df["label"].astype(int)

# ----------------------------
# FEATURE: COUNTRY PRESENCE
# ----------------------------
def has_country(text):
	return int(bool(COUNTRY_REGEX.search(text)))

df["has_country"] = df["article"].apply(has_country)

# ----------------------------
# P(has_country | label)
# ----------------------------
p_country_given_label = (
	df.groupby("label")["has_country"]
	.mean()
)

print("\n--- P(has_country | label) ---")
print(p_country_given_label)

# ----------------------------
# P(label | has_country = 1)
# ----------------------------
label_dist_if_country = (
	df[df["has_country"] == 1]["label"]
	.value_counts(normalize=True)
	.sort_index()
)

print("\n--- P(label | has_country = 1) ---")
print(label_dist_if_country)

# ----------------------------
# SUPPORT
# ----------------------------
support = df["has_country"].mean()
print("\nSupport P(has_country):", support)



--- P(has_country | label) ---
label
0    0.413771
1    0.111447
2    0.091210
3    0.180315
4    0.149405
5    0.265303
6    0.116699
Name: has_country, dtype: float64

--- P(label | has_country = 1) ---
label
0    0.516928
1    0.062619
2    0.054023
3    0.095468
4    0.067979
5    0.183772
6    0.019210
Name: proportion, dtype: float64

Support P(has_country): 0.2355588334562546
